# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- 1000건마다 `1000.parquet`, `2000.parquet`, ... 형태로 순차 저장
- 중단 후 이어서 크롤링 가능 (기존 파일 자동 감지)
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드

In [ ]:
import os
import time
import random
import requests
import queue
import threading
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
from fake_useragent import UserAgent

# --- 사용자 설정 ---
INITIAL_START_ID = 6657186
END_ID = 1
NUM_THREADS = 10         # 20개는 차단 위험이 높으므로 10개 권장
CHUNK_SIZE = 1000
SAVE_DIR = r"../data/parquet"

DELAY_MIN = 0.7
DELAY_MAX = 1.5
BLOCK_SLEEP = 120        # 차단 시 대기 시간 (초)

# --- 전역 변수 및 동기화 객체 ---
buffer = []
next_chunk_num = 1
data_lock = threading.Lock()      # 버퍼 및 통계용 통합 락
pause_event = threading.Event()   # 차단 시 모든 스레드 일시 정지용
pause_event.set()                 # 기본 상태는 '진행'

task_queue = queue.Queue(maxsize=1000)
ua = UserAgent()
session = requests.Session()      # 커넥션 풀링 사용

stats = {'success': 0, 'empty': 0, 'denied': 0, 'error': 0, 'saved': 0}

# ───────────────────────────────────────────────
# 재시작 정보 계산 (효율화)
# ───────────────────────────────────────────────
def get_parquet_files():
    if not os.path.exists(SAVE_DIR):
        return []
    files = [f for f in os.listdir(SAVE_DIR)
             if f.endswith('.parquet') and f.replace('.parquet', '').isdigit()]
    return sorted(files, key=lambda x: int(x.replace('.parquet', '')))

def get_resume_info():
    files = get_parquet_files()
    if not files:
        return INITIAL_START_ID, 1, set()

    existing_ids = set()
    print("기존 데이터를 로드 중입니다. 잠시만 기다려주세요...")
    for f in files:
        try:
            df_tmp = pd.read_parquet(os.path.join(SAVE_DIR, f), columns=['id'])
            existing_ids.update(df_tmp['id'].values)
        except Exception as e:
            print(f"파일 로드 오류 ({f}): {e}")

    last_file_num = int(files[-1].replace('.parquet', ''))
    next_num = (last_file_num // CHUNK_SIZE) + 1
    
    # 중복되지 않은 최소 ID부터 시작
    current_min = min(existing_ids) if existing_ids else INITIAL_START_ID
    resume_id = int(current_min) - 1
    
    print(f"기존 데이터: {len(existing_ids):,}건 확인.")
    print(f"ID {resume_id}부터 시작합니다. (다음 청크 번호: {next_num})")
    return resume_id, next_num, existing_ids

# ───────────────────────────────────────────────
# 데이터 저장 로직 (Thread-Safe)
# ───────────────────────────────────────────────
def save_chunk_if_needed():
    global buffer, next_chunk_num
    with data_lock:
        if len(buffer) >= CHUNK_SIZE:
            chunk_to_save = buffer[:CHUNK_SIZE]
            buffer = buffer[CHUNK_SIZE:]
            
            filename = f'{next_chunk_num * CHUNK_SIZE}.parquet'
            save_path = os.path.join(SAVE_DIR, filename)
            pd.DataFrame(chunk_to_save).to_parquet(save_path, index=False)
            
            stats['saved'] = next_chunk_num
            next_chunk_num += 1

def save_remaining():
    global buffer
    with data_lock:
        if not buffer:
            return
        # 마지막 남은 데이터는 현재까지의 총 수집량을 이름으로 저장
        total_count = stats['success']
        filename = f'final_{total_count}.parquet'
        pd.DataFrame(buffer).to_parquet(os.path.join(SAVE_DIR, filename), index=False)
        print(f"\n[완료] 남은 {len(buffer)}건 저장 완료: {filename}")
        buffer = []

# ───────────────────────────────────────────────
# API 요청 (id 파라미터 최적화)
# ───────────────────────────────────────────────
def fetch_api_data(post_id):
    # 단일 ID 조회 시 tags=id:{id} 방식이 더 안정적임
    url = "https://safebooru.org/index.php"
    params = {
        'page': 'dapi',
        's': 'post',
        'q': 'index',
        'tags': f'id:{post_id}'
    }
    headers = {'User-Agent': ua.random}
    
    try:
        response = session.get(url, params=params, headers=headers, timeout=15)

        if response.status_code in [403, 429]:
            return None, 'denied'
        if response.status_code != 200:
            return None, 'error'

        root = ET.fromstring(response.content)
        post = root.find('post')
        if post is None:
            return None, 'empty'

        file_url = post.get('file_url', '')
        if file_url and not file_url.startswith('http'):
            file_url = 'https:' + file_url
            
        return {
            'id': int(post.get('id')),
            'tags': post.get('tags'),
            'file_url': file_url,
            'sample_url': post.get('sample_url', ''),
            'width': int(post.get('width', 0)),
            'height': int(post.get('height', 0)),
        }, 'success'
    except Exception:
        return None, 'error'

# ───────────────────────────────────────────────
# 워커 스레드
# ───────────────────────────────────────────────
def worker(pbar, existing_ids):
    while True:
        pause_event.wait()  # 차단 시 여기서 모든 스레드가 멈춤
        
        post_id = task_queue.get()
        if post_id is None:
            task_queue.task_done()
            break

        if post_id in existing_ids:
            pbar.update(1)
            task_queue.task_done()
            continue

        result, status = fetch_api_data(post_id)

        if status == 'denied':
            with data_lock:
                stats['denied'] += 1
            
            # 차단 감지 시 전역 정지 트리거
            if pause_event.is_set():
                pause_event.clear() # 다른 스레드들을 멈춤
                print(f"\n[!] 차단 감지(ID: {post_id}). {BLOCK_SLEEP}초간 대기합니다...")
                time.sleep(BLOCK_SLEEP)
                pause_event.set()   # 대기 후 재개
            
            task_queue.put(post_id) # 현재 ID 재시도 위해 큐에 복귀
            task_queue.task_done()
            continue

        with data_lock:
            stats[status] += 1
            if result:
                buffer.append(result)

        save_chunk_if_needed()
        
        pbar.update(1)
        pbar.set_postfix({
            '성공': stats['success'],
            '차단': stats['denied'],
            '오류': stats['error'],
            '버퍼': len(buffer)
        }, refresh=False)
        
        task_queue.task_done()
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

# ───────────────────────────────────────────────
# 메인 실행
# ───────────────────────────────────────────────
if __name__ == "__main__":
    os.makedirs(SAVE_DIR, exist_ok=True)
    
    current_start_id, next_chunk_num, existing_ids = get_resume_info()
    total_tasks = current_start_id - END_ID + 1

    print(f"=== 수집 시작: {current_start_id} -> {END_ID} ===")
    pbar = tqdm(total=total_tasks, desc="진행", ncols=100)

    threads = []
    for _ in range(NUM_THREADS):
        t = threading.Thread(target=worker, args=(pbar, existing_ids))
        t.daemon = True
        t.start()
        threads.append(t)

    try:
        for pid in range(current_start_id, END_ID - 1, -1):
            task_queue.put(pid)
        task_queue.join()
    except KeyboardInterrupt:
        print("\n[!] 중단됨. 데이터를 저장합니다...")

    # 종료 처리
    for _ in range(NUM_THREADS):
        task_queue.put(None)
    for t in threads:
        t.join()

    save_remaining()
    pbar.close()
    print("\n작업이 종료되었습니다.")

저장 경로: c:\Users\EL069\Project\safebooru\data\parquet
=== 크롤링 시작 (ID: 6657186 → 1, 스레드: 20, 딜레이: 0.5~1.0s) ===


진행:   0%|                   | 2327/6657186 [03:13<160:25:09, 11.52it/s, 수집=967, 없음=2, 차단=0, 오류=1357, 파일=0건]  1.25it/s]


[!] 중단 요청. 남은 버퍼를 저장하고 종료합니다.


진행:   0%|                | 2443/6657186 [03:24<74:53:56, 24.68it/s, 수집=1024, 없음=2, 차단=0, 오류=1417, 파일=1000건]